# Surface Crack Detection - Fine-Tuning
Load existing model weights and fine-tune with a lower learning rate.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
DATASET_PATH = '/kaggle/input/datasets/yidazhang07/bridge-cracks-image'
WEIGHTS_PATH = '/kaggle/input/your-dataset-name/model_weights.pth' # Update this!

BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-4  # Lower learning rate for fine-tuning
SAVE_PATH = 'finetuned_weights.pth'

if 'KAGGLE_URL_BASE' in os.environ:
    print('Running on Kaggle')
elif 'COLAB_GPU' in os.environ:
    print('Running on Google Colab')
else:
    print('Running Locally')

In [ ]:
# Data Loading
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

try:
    dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    num_classes = len(dataset.classes)
except Exception as e:
    print('Dataset not loaded:', e)
    num_classes = 4


In [ ]:
# Model Definition (Must match training architecture exactly)
model = models.mobilenet_v2(pretrained=False) # No need for pretrained ImageNet weights
model.classifier[1] = nn.Linear(model.last_channel, num_classes)

# Load Weights
try:
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    print('Successfully loaded weights!')
except Exception as e:
    print(f'Error loading weights from {WEIGHTS_PATH}. Make sure the path is correct.')
    print(e)

model = model.to(device)

# Optimizer with lower learning rate
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [ ]:
# Fine-tuning Loop
best_acc = 0.0
for epoch in range(EPOCHS):
    if 'train_loader' not in locals():
        break
        
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    # Validation
    model.eval()
    v_correct = 0
    v_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            v_total += labels.size(0)
            v_correct += predicted.eq(labels).sum().item()
            
    val_acc = 100. * v_correct / v_total
    print(f'Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%')
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)


In [ ]:
# Download Fine-tuned Weights
import os
if os.path.exists(SAVE_PATH):
    if 'KAGGLE_URL_BASE' in os.environ:
        from IPython.display import FileLink
        display(FileLink(SAVE_PATH))
    elif 'COLAB_GPU' in os.environ:
        from google.colab import files
        files.download(SAVE_PATH)
